# Kaggle Full 01 - Data Preparation (Self-Contained)

This notebook is fully self-contained for Kaggle. It does not import project-local Python modules.

It performs:
1. Load annotation file
2. Normalize labels
3. Crop face with bbox (if available)
4. Resize to 80x80
5. Split and export manifests


In [ ]:
# !pip install -q opencv-python pandas numpy tqdm

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Any
import json

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
@dataclass
class PrepConfig:
    dataset_root: str
    annotation_path: str
    image_rel_col: str = 'image_path'
    label_col: str = 'label'
    split_col: str | None = None
    bbox_cols_xywh: tuple[str, str, str, str] | None = ('bbox_x', 'bbox_y', 'bbox_w', 'bbox_h')
    image_size: int = 80
    bbox_margin_ratio: float = 0.15
    live_values: tuple[Any, ...] = (1, '1', 'live', 'real', True)
    spoof_values: tuple[Any, ...] = (0, '0', 'spoof', 'fake', False)


def load_table(path: str) -> pd.DataFrame:
    p = Path(path)
    suffix = p.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(p)
    if suffix in {'.json', '.jsonl', '.ndjson'}:
        return pd.read_json(p, lines=suffix in {'.jsonl', '.ndjson'})
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(p)
    if suffix == '.txt':
        # Generic whitespace-delimited fallback.
        return pd.read_csv(p, sep=r'\s+', header=None, engine='python')
    raise ValueError(f'Unsupported annotation format: {suffix}')


def parse_celeba_spoof_label_file(path: str | Path, split_name: str) -> pd.DataFrame:
    p = Path(path)
    rows = []
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        image_rel = parts[0]
        numeric = [float(x) for x in parts[1:] if x.replace('.', '', 1).replace('-', '', 1).isdigit()]
        if not numeric:
            continue

        label_raw = numeric[0]
        item = {
            'image_path': image_rel,
            'label': int(label_raw),
            'split': split_name,
        }

        if len(numeric) >= 5:
            item['bbox_x'] = float(numeric[1])
            item['bbox_y'] = float(numeric[2])
            item['bbox_w'] = float(numeric[3])
            item['bbox_h'] = float(numeric[4])

        rows.append(item)

    return pd.DataFrame(rows)


def normalize_label(value: Any, live_values: tuple[Any, ...], spoof_values: tuple[Any, ...]) -> int:
    if value in live_values:
        return 1
    if value in spoof_values:
        return 0

    text = str(value).strip().lower()
    live_set = {str(item).strip().lower() for item in live_values}
    spoof_set = {str(item).strip().lower() for item in spoof_values}

    if text in live_set:
        return 1
    if text in spoof_set:
        return 0

    raise ValueError(f'Unknown label value: {value}')


def clamp_bbox_xyxy(x1: int, y1: int, x2: int, y2: int, h: int, w: int) -> tuple[int, int, int, int]:
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(x1 + 1, min(x2, w))
    y2 = max(y1 + 1, min(y2, h))
    return x1, y1, x2, y2


def expand_bbox_xywh(x: float, y: float, bw: float, bh: float, h: int, w: int, margin_ratio: float):
    x1 = int(round(x))
    y1 = int(round(y))
    x2 = int(round(x + bw))
    y2 = int(round(y + bh))

    mx = int((x2 - x1) * margin_ratio)
    my = int((y2 - y1) * margin_ratio)
    return clamp_bbox_xyxy(x1 - mx, y1 - my, x2 + mx, y2 + my, h, w)


def create_split_manifests(df: pd.DataFrame, train_ratio: float = 0.8, val_ratio: float = 0.1, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    rng.shuffle(idx)
    shuffled = df.iloc[idx].reset_index(drop=True)

    train_end = int(len(shuffled) * train_ratio)
    val_end = train_end + int(len(shuffled) * val_ratio)
    return {
        'train': shuffled.iloc[:train_end].reset_index(drop=True),
        'val': shuffled.iloc[train_end:val_end].reset_index(drop=True),
        'test': shuffled.iloc[val_end:].reset_index(drop=True),
    }




In [ ]:
# Kaggle CelebA-Spoof dataset locations (fixed paths)
KAGGLE_INPUT_ROOT = Path('/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing')
DATASET_BASE = KAGGLE_INPUT_ROOT / 'CelebA_Spoof_' / 'CelebA_Spoof'

if not DATASET_BASE.exists():
    raise FileNotFoundError(f'Could not find CelebA-Spoof root at {DATASET_BASE}')

DATASET_ROOT = DATASET_BASE
TRAIN_LABEL_PATH = DATASET_BASE / 'metas' / 'intra_test' / 'train_label.txt'
TEST_LABEL_PATH = DATASET_BASE / 'metas' / 'intra_test' / 'test_label.txt'

OUTPUT_ROOT = Path('/kaggle/working/celeba_spoof_prepared_full')
CROP_ROOT = OUTPUT_ROOT / 'crops_80x80'
MANIFEST_ROOT = OUTPUT_ROOT / 'manifests'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CROP_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

config = PrepConfig(
    dataset_root=str(DATASET_ROOT),
    annotation_path=str(TRAIN_LABEL_PATH),  # kept for compatibility with helper functions
    image_rel_col='image_path',
    label_col='label',
    split_col='split',
    bbox_cols_xywh=('bbox_x', 'bbox_y', 'bbox_w', 'bbox_h'),
    image_size=80,
    bbox_margin_ratio=0.15,
)

print('DATASET_BASE:', DATASET_BASE)
print('DATASET_ROOT:', DATASET_ROOT)
print('TRAIN_LABEL_PATH:', TRAIN_LABEL_PATH)
print('TEST_LABEL_PATH:', TEST_LABEL_PATH)
print(config)




In [ ]:
train_table = parse_celeba_spoof_label_file(TRAIN_LABEL_PATH, split_name='train')
test_table = parse_celeba_spoof_label_file(TEST_LABEL_PATH, split_name='test')
table = pd.concat([train_table, test_table], ignore_index=True)

print('Rows:', len(table))
print('Columns:', list(table.columns))
print('Split counts:')
print(table['split'].value_counts())
display(table.head(3))



In [ ]:
required = [config.image_rel_col, config.label_col]
for col in required:
    if col not in table.columns:
        raise KeyError(f'Missing required column: {col}')

has_split = bool(config.split_col and config.split_col in table.columns)
has_bbox = bool(config.bbox_cols_xywh and all(col in table.columns for col in config.bbox_cols_xywh))

print('has_split:', has_split)
print('has_bbox:', has_bbox)

In [ ]:
def build_normalized_rows(source_df: pd.DataFrame):
    rows = []
    for row in source_df.itertuples(index=False):
        label = normalize_label(getattr(row, config.label_col), config.live_values, config.spoof_values)
        item = {
            'image_rel': str(getattr(row, config.image_rel_col)),
            'label': int(label),
        }
        if has_bbox:
            bx, by, bw, bh = config.bbox_cols_xywh
            item['bbox_x'] = float(getattr(row, bx))
            item['bbox_y'] = float(getattr(row, by))
            item['bbox_w'] = float(getattr(row, bw))
            item['bbox_h'] = float(getattr(row, bh))
        rows.append(item)
    return pd.DataFrame(rows)

if has_split:
    split_values = sorted(table[config.split_col].dropna().unique().tolist())
    print('split values:', split_values)
else:
    split_values = []


In [ ]:
def resolve_image_path(image_rel: str) -> Path:
    rel = Path(str(image_rel).lstrip('/'))
    candidates = [
        Path(config.dataset_root) / rel,
        Path(config.dataset_root) / 'Data' / rel,
    ]

    rel_parts = rel.parts
    if rel_parts and rel_parts[0] == 'Data':
        rel_wo_data = Path(*rel_parts[1:]) if len(rel_parts) > 1 else Path('')
        if str(rel_wo_data):
            candidates.extend([
                Path(config.dataset_root) / rel_wo_data,
                Path(config.dataset_root) / 'Data' / rel_wo_data,
            ])

    for c in candidates:
        if c.exists():
            return c

    return candidates[0]


def crop_rows(df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for i, row in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc='Cropping')):
        src = resolve_image_path(row.image_rel)
        image = cv2.imread(str(src))
        if image is None:
            continue

        if has_bbox:
            x1, y1, x2, y2 = expand_bbox_xywh(
                row.bbox_x,
                row.bbox_y,
                row.bbox_w,
                row.bbox_h,
                h=image.shape[0],
                w=image.shape[1],
                margin_ratio=config.bbox_margin_ratio,
            )
            face = image[y1:y2, x1:x2]
        else:
            x1, y1, x2, y2 = 0, 0, image.shape[1], image.shape[0]
            face = image

        face = cv2.resize(face, (config.image_size, config.image_size), interpolation=cv2.INTER_LINEAR)
        name = f'{i:08d}_{Path(row.image_rel).stem}.jpg'
        dst = out_dir / name
        cv2.imwrite(str(dst), face)

        records.append(
            {
                'image_path': str(dst),
                'label': int(row.label),
                'bbox_xyxy': [int(x1), int(y1), int(x2), int(y2)],
            }
        )

    return pd.DataFrame(records)



In [ ]:
saved = {}
all_frames = []

if has_split:
    split_values = {str(v).strip().lower() for v in table[config.split_col].dropna().unique().tolist()}
    has_val = 'val' in split_values

    if has_val:
        split_map = {'train': 'train', 'val': 'val', 'test': 'test'}
        for out_name, split_value in split_map.items():
            split_df = table[table[config.split_col] == split_value].reset_index(drop=True)
            norm_df = build_normalized_rows(split_df)
            cropped_df = crop_rows(norm_df, CROP_ROOT / out_name)

            out_path = MANIFEST_ROOT / f'{out_name}.csv'
            cropped_df.to_csv(out_path, index=False)
            saved[out_name] = out_path
            all_frames.append(cropped_df)
    else:
        # CelebA-Spoof intra_test provides only train/test. Create val from train.
        train_df = table[table[config.split_col] == 'train'].reset_index(drop=True)
        test_df = table[table[config.split_col] == 'test'].reset_index(drop=True)

        train_norm = build_normalized_rows(train_df)
        train_splits = create_split_manifests(train_norm, train_ratio=0.9, val_ratio=0.1, seed=42)

        for name in ['train', 'val']:
            split_df = train_splits[name]
            cropped_df = crop_rows(split_df, CROP_ROOT / name)
            out_path = MANIFEST_ROOT / f'{name}.csv'
            cropped_df.to_csv(out_path, index=False)
            saved[name] = out_path
            all_frames.append(cropped_df)

        test_norm = build_normalized_rows(test_df)
        test_cropped = crop_rows(test_norm, CROP_ROOT / 'test')
        out_path = MANIFEST_ROOT / 'test.csv'
        test_cropped.to_csv(out_path, index=False)
        saved['test'] = out_path
        all_frames.append(test_cropped)
else:
    norm_df = build_normalized_rows(table)
    cropped_df = crop_rows(norm_df, CROP_ROOT)

    split_frames = create_split_manifests(cropped_df, train_ratio=0.8, val_ratio=0.1, seed=42)
    for name, frame in split_frames.items():
        out_path = MANIFEST_ROOT / f'{name}.csv'
        frame.to_csv(out_path, index=False)
        saved[name] = out_path
    all_frames.append(cropped_df)

merged = pd.concat(all_frames, ignore_index=True)
print('Saved manifests:')
for k, v in saved.items():
    print(' ', k, '->', v)

print('Label distribution:')
print(merged['label'].value_counts().sort_index())
